# GRID3 / CIESIN: 
## Storing & Serving CIESIN Data using the Cloudflare Network
*June 17, 2026*

### Storage: What is Cloudflare R2?

A remote object store that stores flattened data+metadata 

R2 is fully AWS S3-spec compliant, meaning existing tools and methods for operating with S3 data stores can be applied directly to R2.
(Migrating between vendors, if necessary, is straightforward)

### Why storage buckets over other data hosting methods, like Google Drive, or Dropbox?

#### Object vs File storage

*Source: https://cloud.google.com/learn/what-is-object-storage*

##### **File storage**
File storage stores and organizes data into folders, similar to the physical files you might store in a paper filing system in an office. If you need information from a file, you’ll need to know what room, cabinet, drawer, and folder contains that specific document. This same hierarchical storage structure is used for file storage, where files are named, tagged with metadata, and then placed in folders. 

To locate a piece of data, you’ll need to know the correct path to find it. Over time, searching and retrieving data files can become time-consuming as the number of files grows. While scalability is more limited, it is a simple way to store small amounts of just about any type of data and make it accessible to multiple users at once. 

##### **Block storage**
Block storage improves on the performance of file storage, breaking files into separate blocks and storing them separately. A block-storage system will assign a unique identifier to each chunk of raw data, which can then be used to reassemble them into the complete file when you need to access it. Block storage doesn’t require a single path to data, so you can store it wherever is most convenient and still retrieve it quickly when needed. 

Block storage works well for organizations that work with large amounts of transactional data or mission-critical applications that need minimal delay and consistent performance. However, it can be expensive, offers no metadata capabilities, and requires an operating system to access blocks. 

##### **Object storage**
Object storage, as discussed earlier, saves files in a flat data environment, or storage pool, as a self-contained object that contains all the data, a unique identifier, and detailed metadata that contains information about the data, permissions, policies, and other contingencies. Object storage works best for static storage, especially for unstructured data, where you write data once but may need to read it many times. 

While object storage eliminates the need for directories, folders, and other complex hierarchical organization, it’s not a good solution for dynamic data that is changing constantly as you’ll need to rewrite the entire object to modify it. In some cases, file storage and block storage may still suit your needs depending on your speed and performance requirements.  



### Why use Cloudflare over other established vendors, i.e., AWS?

`$$$`

### What is currently stored in the CIESIN R2 buckets?
(V0 framework)

#### `ciesin-dev/`
dev.ciesin.app/
#### `ciesin-prod/`
prod.ciesin.app/

- `comms/` <-- web assets for public-facing sites
- `data/` <-- cloud-optimized geospatial data for direct data access via API
- `maps/` <-- static images fetched by the map archive ()
- `stac/` <-- JSON metadata associated with the map archive 
- `tiles/` <-- vector + raster tiles for the CIESIN webmap (tiles.ciesin.columbia.edu)

The above are generally organized by `project/theme/` and/or `project/iso3/`

### How do I get my data into these bucket(s)?

#### Web dashboard 

#### rclone CLI

- Similar to 

# Cloud Sync

rclone operations tailored for CIESIN dev/prod R2 buckets

if rclone is not configured, run `rclone config` and create a new profile using relevant bucket credentials

**overview:**
1. Edit `DEV_UPLOADS` manifest to define files for a new dataset
2. Run *Upload to dev* cells with `DRY_RUN = True` to preview commands
3. Set `DRY_RUN = False` and re-run to execute
4. Use *Dev -> Prod sync* cells when promoting staged files to production environment

In [1]:
import os
import subprocess
import shlex
import sys
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # loads .env from cwd or any parent directory

REMOTE      = os.environ["RCLONE_REMOTE"]
DEV_BUCKET  = f"{REMOTE}:{os.environ['DEV_BUCKET_NAME']}"
PROD_BUCKET = f"{REMOTE}:{os.environ['PROD_BUCKET_NAME']}"

# NB: Safety default. flip to False only when ready to execute actual remote operations
DRY_RUN = False

SCRATCH = Path(os.getenv("SCRATCH_DIR", "/tmp/cloudtools"))

# logs
LOG_DIR = Path(os.getenv("LOG_DIR", "/tmp/cloudtools/logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
SESSION_LOG = LOG_DIR / f"cloud_sync_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
print(f"Session log: {SESSION_LOG}")

Session log: /tmp/cloudtools/logs/cloud_sync_20260609_191602.log


## CLI presets

These profiles of rclone flags cover the current main transfer scenarios.
Pass them into any config manifest (below) via the `flags=` argument to override defaults.

In [2]:
def flags_to_args(flags: dict) -> list:
    """Convert {flag: value} dict to a flat rclone arg list. None -> boolean flag."""
    args = []
    for k, v in flags.items():
        args.append(k)
        if v is not None:
            args.append(v)
    return args


# Multi-GB: few parallel streams, large chunks
LARGE_FILE_FLAGS = {
    "--s3-chunk-size":        "256M",
    "--s3-upload-concurrency": "8",
    "--multi-thread-streams": "8",
    "--multi-thread-cutoff":  "64M",
    "--buffer-size":          "64M",
    "--transfers":            "4",
    "--checkers":             "8",
    "--fast-list":            None,
    "--retries":              "3",
    "--progress":             True,
    # "--stats":                "30s",
    "--s3-no-check-bucket":   None,
}

# Many small files: STAC JSON, metadata, map images
SMALL_FILES_FLAGS = {
    "--transfers":          "32",
    "--checkers":           "32",
    "--fast-list":          None,
    "--size-only":          None,
    # "--stats":              "30s",
    "--s3-no-check-bucket": None,
}

# R2->R2 bucket sync: server-side copy, skip checksum (avoid R2 hash charges)
R2_SYNC_FLAGS = {
    "--s3-chunk-size":        "256M",
    "--s3-upload-concurrency": "8",
    "--fast-list":            None,
    "--size-only":            None,
    "--transfers":            "4",
    "--checkers":             "8",
    # "--stats":                "30s",
    "--s3-no-check-bucket":   None,
}

## Helpers

In [3]:
def run_rclone(cmd: list, dry_run: bool = DRY_RUN) -> None:
    """Run an rclone command, streaming merged stdout+stderr to cell output and SESSION_LOG."""
    if dry_run:
        cmd = cmd + ["--dry-run"]
    cmd_str = " ".join(shlex.quote(str(a)) for a in cmd)
    header = f"\n[{datetime.now().isoformat(timespec='seconds')}] $ {cmd_str}"
    print(header)
    with SESSION_LOG.open("a") as log:
        log.write(header + "\n")
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        proc.wait()
        if proc.returncode != 0:
            msg = f"rclone exited {proc.returncode}\n"
            log.write(msg)
            raise RuntimeError(f"rclone exited {proc.returncode}")


def upload_pmtiles(local: str, remote: str, flags: dict = None, dry_run: bool = DRY_RUN) -> None:
    if flags is None:
        flags = LARGE_FILE_FLAGS
    run_rclone(
        ["rclone", "copyto", local, remote,
         "--header-upload", "Content-Type: application/vnd.pmtiles",
         *flags_to_args(flags)],
        dry_run=dry_run,
    )


def upload_json(local: str, remote: str, flags: dict = None, dry_run: bool = DRY_RUN) -> None:
    if flags is None:
        flags = SMALL_FILES_FLAGS
    run_rclone(
        ["rclone", "copyto", local, remote,
         "--header-upload", "Content-Type: application/json",
         *flags_to_args(flags)],
        dry_run=dry_run,
    )

## Data manifest

Edit `DEV_UPLOADS` to define a new batch of files to stage in `ciesin-dev`.
Each entry needs `local`, `remote`, `content_type`, and optionally a `flags` override to set a custom rclone preset.

In [ ]:
DEV_UPLOADS = [
    # # --- Base / Protomaps ---
    # {
    #     "local":        str(SCRATCH / "tiles/base.pmtiles"),
    #     "remote":       f"{DEV_BUCKET}/tiles/base.pmtiles",
    #     "content_type": "application/vnd.pmtiles",
    #     "flags":        LARGE_FILE_FLAGS,
    # },
    # # --- Overture ---
    # {
    #     "local":        str(SCRATCH / "tiles/overture/buildings.pmtiles"),
    #     "remote":       f"{DEV_BUCKET}/tiles/overture/buildings.pmtiles",
    #     "content_type": "application/vnd.pmtiles",
    #     "flags":        LARGE_FILE_FLAGS,
    # },
    # {
    #     "local":        str(SCRATCH / "tiles/overture/transportation.pmtiles"),
    #     "remote":       f"{DEV_BUCKET}/tiles/overture/transportation.pmtiles",
    #     "content_type": "application/vnd.pmtiles",
    #     "flags":        LARGE_FILE_FLAGS,
    # },
    # # --- GRID3 COD ---
    # {
    #     "local":        str(SCRATCH / "tiles/grid3/cod/GRID3_COD_settlement_extents_v3_1.pmtiles"),
    #     "remote":       f"{DEV_BUCKET}/tiles/grid3/cod/GRID3_COD_settlement_extents_v3_1.pmtiles",
    #     "content_type": "application/vnd.pmtiles",
    #     "flags":        LARGE_FILE_FLAGS,
    # },
    # {
    #     "local":        str(SCRATCH / "tiles/grid3/cod/GRID3_COD_health_areas_v8_0.pmtiles"),
    #     "remote":       f"{DEV_BUCKET}/tiles/grid3/cod/GRID3_COD_health_areas_v8_0.pmtiles",
    #     "content_type": "application/vnd.pmtiles",
    #     "flags":        LARGE_FILE_FLAGS,
    # },
    # # --- GRID3 NGA ---
    # {
    #     "local":        str(SCRATCH / "tiles/grid3/nga/GRID3_NGA_settlement_extents_v4_0.pmtiles"),
    #     "remote":       f"{DEV_BUCKET}/tiles/grid3/nga/GRID3_NGA_settlement_extents_v4_0.pmtiles",
    #     "content_type": "application/vnd.pmtiles",
    #     "flags":        LARGE_FILE_FLAGS,
    # },
        # --- GRID3 NGA COG ---
    {
        "local":        "/mnt/d/mheaton/cloudflare/ciesin-dev/COG/GRID3 NGA - Travel Time Friction Surface v1.0/GRID3_NGA_walk_travel_time_friction_surface_v1_0_cog.tif",
        "remote":       f"{DEV_BUCKET}/data/grid3/nga/friction_surface/GRID3_NGA_walk_travel_time_friction_surface_v1_0.tif",
        "content_type": "application=geotiff",
        "flags":        LARGE_FILE_FLAGS,
    },
        # --- GRID3 COD COG ---
    {
        "local":        "/mnt/d/mheaton/cloudflare/ciesin-dev/COG/GRID3 COD - Travel Time Friction Surface v1.0/GRID3_COD_mix_travel_time_friction_surface_v1_cog.tif",
        "remote":       f"{DEV_BUCKET}/data/grid3/cod/friction_surface/GRID3_COD_mix_travel_time_friction_surface_v1.tif",
        "content_type": "application=geotiff",
        "flags":        LARGE_FILE_FLAGS,
    }
]

## Upload local -> dev

Uploads every entry in `DEV_UPLOADS`. Skips files that don't exist locally.
Run with `DRY_RUN = True` first to preview the exact commands.

In [ ]:
for entry in DEV_UPLOADS:
    local = entry["local"]
    if not Path(local).exists():
        print(f"SKIP (not found): {local}")
        continue
    print(f"\n{'='*60}\n{Path(local).name}")
    run_rclone(
        ["rclone", "copyto", local, entry["remote"],
         "--header-upload", f"Content-Type: {entry['content_type']}",
         *flags_to_args(entry.get("flags", LARGE_FILE_FLAGS))],
    )

## Dev -> Prod sync

Promotes content from `ciesin-dev` to `ciesin-prod` using R2 server-side copy
(S3 CopyObject API: no local bandwidth).

Set `SYNC_PREFIXES` to specific subpaths to sync a subset, or use a top-level
key (`tiles`, `stac`, `maps`, `data`) for a full section sync.

In [ ]:
SYNC_PREFIXES = [
    "tiles/grid3/cod",
    "tiles/grid3/nga",
    "tiles/overture",
    # "tiles",
]

for prefix in SYNC_PREFIXES:
    print(f"\n{'='*60}\nSyncing: {prefix}")
    run_rclone([
        "rclone", "sync",
        f"{DEV_BUCKET}/{prefix}/",
        f"{PROD_BUCKET}/{prefix}",
        *flags_to_args(R2_SYNC_FLAGS),
    ])

### STAC / static map sync

Syncs `stac/` and `maps/` separately using the small-files profile
(high `--transfers`, no multipart overhead). Safe to run independently
from tile syncs: STAC often updates without tile changes.

In [ ]:
for prefix in ["stac", "maps"]:
    print(f"\n{'='*60}\nSyncing: {prefix}")
    run_rclone([
        "rclone", "sync",
        f"{DEV_BUCKET}/{prefix}/",
        f"{PROD_BUCKET}/{prefix}",
        "--header-upload", "Content-Type: application/json",
        *flags_to_args(SMALL_FILES_FLAGS),
    ])

## Verification

Compares dev and prod for a given prefix. Writes diff/missing reports
to the current directory with a timestamp suffix.
`rclone check` is always read-only, so `dry_run=False` is safe here.

In [4]:
CHECK_PREFIX = "tiles/grid3"
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

run_rclone(
    [
        "rclone", "check",
        f"{DEV_BUCKET}/{CHECK_PREFIX}",
        f"{PROD_BUCKET}/{CHECK_PREFIX}",
        "--differ",         str(LOG_DIR / f"diff_{ts}.txt"),
        "--missing-on-dst", str(LOG_DIR / f"missing_prod_{ts}.txt"),
        "--missing-on-src", str(LOG_DIR / f"missing_dev_{ts}.txt"),
        "--match",          str(LOG_DIR / f"match_{ts}.txt"),
        "--fast-list",
    ],
    dry_run=False,  # check is read-only
)
print(f"Check reports written to {LOG_DIR}/")


[2026-06-04T15:33:43] $ rclone check ciesin-r2:ciesin-dev/tiles/grid3 ciesin-r2:ciesin-prod/tiles/grid3 --differ /tmp/cloudtools/logs/diff_20260604_153343.txt --missing-on-dst /tmp/cloudtools/logs/missing_prod_20260604_153343.txt --missing-on-src /tmp/cloudtools/logs/missing_dev_20260604_153343.txt --match /tmp/cloudtools/logs/match_20260604_153343.txt --fast-list
2026/06/04 15:33:44 ERROR : cod/GRID3_COD_antenne_v8_0.pmtiles: file not in S3 bucket ciesin-prod path tiles/grid3
2026/06/04 15:33:44 ERROR : cod/GRID3_COD_antenne_v8_0_centroids.pmtiles: file not in S3 bucket ciesin-prod path tiles/grid3
2026/06/04 15:33:44 ERROR : cod/GRID3_COD_provinces_v8_0.pmtiles: file not in S3 bucket ciesin-prod path tiles/grid3
2026/06/04 15:33:44 ERROR : cod/GRID3_COD_provinces_v8_0_centroids.pmtiles: file not in S3 bucket ciesin-prod path tiles/grid3
2026/06/04 15:33:44 ERROR : cod/temp-2/GRID3_COD_health_areas_v8_0.pmtiles: file not in S3 bucket ciesin-prod path tiles/grid3
2026/06/04 15:33:44 E

RuntimeError: rclone exited 1

## Fallback: stage to local machine then push to remote

For files where R2->R2 multipart copy fails or stalls (rare, typically >100 GB
edge cases), this cell downloads via rclone to `/tmp`, uploads, then cleans up.
Only use this when the direct R2->R2 path is confirmed to be failing.

In [ ]:
FALLBACK_REMOTE_SRC = f"{DEV_BUCKET}/tiles/base.pmtiles"
FALLBACK_REMOTE_DST = f"{PROD_BUCKET}/tiles/base.pmtiles"
FALLBACK_LOCAL      = Path("/tmp/rclone_fallback_base.pmtiles")

if DRY_RUN:
    print(f"DRY RUN: would download {FALLBACK_REMOTE_SRC}")
    print(f"         upload to      {FALLBACK_REMOTE_DST}")
    print(f"         then delete    {FALLBACK_LOCAL}")
else:
    print(f"Step 1/3: download {FALLBACK_REMOTE_SRC} -> {FALLBACK_LOCAL}")
    run_rclone(
        ["rclone", "copyto", FALLBACK_REMOTE_SRC, str(FALLBACK_LOCAL),
         *flags_to_args(LARGE_FILE_FLAGS)],
        dry_run=False,
    )

    print(f"\nStep 2/3: upload {FALLBACK_LOCAL} -> {FALLBACK_REMOTE_DST}")
    upload_pmtiles(str(FALLBACK_LOCAL), FALLBACK_REMOTE_DST, dry_run=False)

    FALLBACK_LOCAL.unlink(missing_ok=True)
    print(f"\nStep 3/3: removed {FALLBACK_LOCAL}")